# Exploratory decay-component analysis

This preserved research draft explores fixed two-state lifetime components, residual background, MPET, and intensity relationships. It is not yet a supported batch workflow. Review the assumptions and edit the data-directory cell before running.

In [ ]:
import sys
import scipy
import iflip3

print(sys.executable)
print("SciPy:", scipy.__version__)
print("iFLiP3 ready")

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from iflip3 import (
    average_background,
    calculate_mpet,
    read_iflip3,
)

AFTERPULSE_RATIO = 0.03
SPC_RANGE = (0.4, 12.3)

In [ ]:
data_dir = Path(r"Z:\FLIM FLIP\MOUSE_DATE_FOLDER")

background_files = sorted(data_dir.glob("*_BG*.iFLiP3"))
measurement_files = sorted(
    path
    for path in data_dir.glob("*.iFLiP3")
    if "_BG" not in path.name
)

print("Background files:")
for path in background_files:
    print(" ", path.name)

print("\nMeasurement files:")
for path in measurement_files:
    print(" ", path.name)

backgrounds = [read_iflip3(path) for path in background_files]
recordings = [read_iflip3(path) for path in measurement_files]

background_curve = average_background(backgrounds)

print("\nLoaded shapes:")
for recording in recordings:
    print(recording.path.name, recording.data.shape)

In [ ]:
recording_index = 3
recording = recordings[recording_index]

print("Selected:", recording.path.name)
print("Samples:", recording.n_samples)
print("Duration:", recording.n_samples / recording.header.sampling_frequency, "seconds")
print("Lifetime resolution:", recording.header.lifetime_resolution, "ns")
print("Pulse interval:", recording.header.pulse_interval_ns, "ns")
print("Header t0:", recording.header.get_path("state.t0.Value"), "ns")

In [ ]:
afterpulse_ratio = 0.03
spc_range = (0.4, 12.3)
t0 = float(recording.header.get_path("state.t0.Value"))

mpet, correction = calculate_mpet(
    recording,
    spc_range=spc_range,
    t0=t0,
    measured_background=background_curve,
    afterpulse_ratio=afterpulse_ratio,
)

raw_intensity = recording.data[:, :, 0].sum(axis=0)
corrected_intensity = correction.corrected[:, :, 0].sum(axis=0)

print("Mean raw photons/sample:", raw_intensity.mean())
print("Mean corrected photons/sample:", corrected_intensity.mean())
print("Mean MPET:", mpet[:, 0].mean(), "ns")
print(
    "Negative corrected bins:",
    100 * np.mean(correction.corrected < 0),
    "%",
)

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(13, 7),
    sharex=True,
    constrained_layout=True,
)

axes[0].plot(
    recording.sample_time,
    raw_intensity,
    color="0.7",
    linewidth=0.8,
    label="Raw",
)

axes[0].plot(
    recording.sample_time,
    corrected_intensity,
    color="tab:blue",
    linewidth=0.8,
    label="Corrected",
)

axes[0].set_ylabel("Photons per sample")
axes[0].set_title(recording.path.name)
axes[0].legend()

axes[1].plot(
    recording.sample_time,
    mpet[:, 0],
    color="tab:orange",
    linewidth=0.8,
)

axes[1].axhline(
    np.nanmean(mpet[:, 0]),
    color="black",
    linestyle="--",
    linewidth=1,
    label="Mean MPET",
)

axes[1].set_xlabel("Recording time (s)")
axes[1].set_ylabel("MPET (ns)")
axes[1].legend()

plt.show()

In [ ]:
raw_mean_decay = recording.data[:, :, 0].mean(axis=1)
corrected_mean_decay = correction.corrected[:, :, 0].mean(axis=1)
measured_background = background_curve[:, 0]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
    constrained_layout=True,
)

axes[0].plot(
    recording.lifetime_time,
    raw_mean_decay,
    label="Raw decay",
)

axes[0].plot(
    recording.lifetime_time,
    corrected_mean_decay,
    label="Corrected decay",
)

axes[0].plot(
    recording.lifetime_time,
    measured_background,
    label="Measured background",
)

axes[0].axvline(t0, color="black", linestyle="--", label="Header t0")
axes[0].set_xlabel("Lifetime time (ns)")
axes[0].set_ylabel("Mean counts per sample")
axes[0].set_title("Linear scale")
axes[0].legend()

axes[1].semilogy(
    recording.lifetime_time,
    np.clip(raw_mean_decay, 1e-3, None),
    label="Raw decay",
)

axes[1].semilogy(
    recording.lifetime_time,
    np.clip(corrected_mean_decay, 1e-3, None),
    label="Corrected decay",
)

axes[1].semilogy(
    recording.lifetime_time,
    np.clip(measured_background, 1e-3, None),
    label="Measured background",
)

axes[1].axvline(t0, color="black", linestyle="--", label="Header t0")
axes[1].set_xlabel("Lifetime time (ns)")
axes[1].set_ylabel("Mean counts per sample")
axes[1].set_title("Log scale")
axes[1].legend()

plt.show()

In [ ]:
from iflip3 import fit_decay

aggregate_decay = correction.corrected[:, :, 0].sum(axis=1)

double_fit = fit_decay(
    recording.lifetime_time,
    aggregate_decay,
    n_components=2,
    pulse_interval=recording.header.pulse_interval_ns,
    initial={
        "tau1": 0.6,
        "tau2": 2.5,
        "t0": float(recording.header.get_path("state.t0.Value")),
        "sigma": float(recording.header.get_path("state.beta6.Value")),
    },
    fixed={
        "background": 0.0,
    },
    weighting="poisson",
)

print("Converged:", double_fit.success)
print("Message:", double_fit.message)
print("Lifetimes:", double_fit.lifetimes, "ns")
print("Amplitudes:", double_fit.amplitudes)
print("t0:", double_fit.t0, "ns")
print("IRF sigma:", double_fit.irf_sigma, "ns")

In [ ]:
pearson_residuals = (
    double_fit.residuals
    / np.sqrt(np.maximum(aggregate_decay, 1))
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(11, 7),
    sharex=True,
    constrained_layout=True,
    height_ratios=[3, 1],
)

axes[0].semilogy(
    recording.lifetime_time,
    np.clip(aggregate_decay, 1, None),
    "o",
    markersize=4,
    label="Measured",
)

axes[0].semilogy(
    recording.lifetime_time,
    np.clip(double_fit.fitted, 1, None),
    color="tab:red",
    linewidth=2,
    label="Double-exponential fit",
)

axes[0].set_ylabel("Aggregated photon counts")
axes[0].legend()

axes[1].axhline(0, color="black", linewidth=1)
axes[1].plot(
    recording.lifetime_time,
    pearson_residuals,
    "o-",
    markersize=3,
)

axes[1].set_xlabel("Lifetime time (ns)")
axes[1].set_ylabel("Residual")

plt.show()

In [ ]:
aggregate_decay = correction.corrected[:, :, 0].sum(axis=1)

fit_mask = (
    (recording.lifetime_time >= 0.4)
    & (recording.lifetime_time <= 12.3)
    & np.isfinite(aggregate_decay)
    & (aggregate_decay > 0)
)

fit_time = recording.lifetime_time[fit_mask]
fit_counts = aggregate_decay[fit_mask]

print("Bins used:", fit_mask.sum())
print("Excluded bins:", (~fit_mask).sum())
print("Minimum fitted count:", fit_counts.min())

In [ ]:
double_fit = fit_decay(
    fit_time,
    fit_counts,
    n_components=2,
    pulse_interval=recording.header.pulse_interval_ns,
    initial={
        "tau1": 0.6,
        "tau2": 2.5,
        "t0": float(recording.header.get_path("state.t0.Value")),
        "sigma": float(recording.header.get_path("state.beta6.Value")),
    },
    fixed={"background": 0.0},
    weighting="poisson",
)

print("Converged:", double_fit.success)
print("Message:", double_fit.message)
print("Lifetimes:", double_fit.lifetimes, "ns")
print("Amplitudes:", double_fit.amplitudes)
print("t0:", double_fit.t0, "ns")
print("IRF sigma:", double_fit.irf_sigma, "ns")

In [ ]:
pearson_residuals = (
    double_fit.residuals
    / np.sqrt(np.maximum(fit_counts, 1))
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(11, 7),
    sharex=True,
    constrained_layout=True,
    height_ratios=[3, 1],
)

axes[0].semilogy(
    fit_time,
    fit_counts,
    "o",
    markersize=4,
    label="Measured",
)

axes[0].semilogy(
    fit_time,
    double_fit.fitted,
    color="tab:red",
    linewidth=2,
    label="Double-exponential fit",
)

axes[0].set_ylabel("Aggregated photon counts")
axes[0].legend()

axes[1].axhline(0, color="black", linewidth=1)
axes[1].plot(fit_time, pearson_residuals, "o-", markersize=3)
axes[1].set_xlabel("Lifetime time (ns)")
axes[1].set_ylabel("Pearson residual")

plt.show()

In [ ]:
double_fit_bg = fit_decay(
    fit_time,
    fit_counts,
    n_components=2,
    pulse_interval=recording.header.pulse_interval_ns,
    initial={
        "tau1": 0.6,
        "tau2": 2.5,
        "t0": float(recording.header.get_path("state.t0.Value")),
        "sigma": float(recording.header.get_path("state.beta6.Value")),
        "background": np.median(fit_counts[-10:]),
    },
    weighting="poisson",
)

print("Converged:", double_fit_bg.success)
print("Lifetimes:", double_fit_bg.lifetimes, "ns")
print("Amplitudes:", double_fit_bg.amplitudes)
print("t0:", double_fit_bg.t0, "ns")
print("IRF sigma:", double_fit_bg.irf_sigma, "ns")
print("Residual background:", double_fit_bg.background)

In [ ]:
pearson = (
    double_fit_bg.residuals
    / np.sqrt(np.maximum(fit_counts, 1))
)

percent_residual = (
    100 * double_fit_bg.residuals
    / np.maximum(fit_counts, 1)
)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(11, 9),
    sharex=True,
    constrained_layout=True,
    height_ratios=[3, 1, 1],
)

axes[0].semilogy(
    fit_time,
    fit_counts,
    "o",
    markersize=4,
    label="Measured",
)

axes[0].semilogy(
    fit_time,
    double_fit_bg.fitted,
    color="tab:red",
    linewidth=2,
    label="Fit with residual background",
)

axes[0].set_ylabel("Photon counts")
axes[0].legend()

axes[1].axhline(0, color="black")
axes[1].plot(fit_time, pearson, "o-", markersize=3)
axes[1].set_ylabel("Pearson residual")

axes[2].axhline(0, color="black")
axes[2].plot(fit_time, percent_residual, "o-", markersize=3)
axes[2].set_ylabel("Residual (%)")
axes[2].set_xlabel("Lifetime time (ns)")

plt.show()

In [ ]:
print("Converged:", double_fit_bg.success)
print("Lifetimes:", double_fit_bg.lifetimes, "ns")
print("Amplitudes:", double_fit_bg.amplitudes)
print("t0:", double_fit_bg.t0, "ns")
print("IRF sigma:", double_fit_bg.irf_sigma, "ns")
print("Residual background:", double_fit_bg.background)

In [ ]:
from iflip3 import bin_curves, fit_global

binned_decay = bin_curves(
    correction.corrected[:, :, 0],
    group_size=10,
    reducer="sum",
)

binned_decay_for_fit = binned_decay[fit_mask, :]

global_fit = fit_global(
    fit_time,
    binned_decay_for_fit,
    n_components=2,
    pulse_interval=recording.header.pulse_interval_ns,
    initial_lifetimes=double_fit_bg.lifetimes,
    initial_t0=double_fit_bg.t0,
    initial_sigma=double_fit_bg.irf_sigma,
    shared_t0=True,
    shared_sigma=True,
    include_background=True,
    nonnegative=True,
    weighting="poisson",
    normalize="area",
    max_nfev=100,
)

print("Converged:", global_fit.success)
print("Message:", global_fit.message)
print("Global lifetimes:", global_fit.lifetimes, "ns")
print("Global t0:", global_fit.recordings[0].t0, "ns")
print("Global IRF sigma:", global_fit.recordings[0].irf_sigma, "ns")

In [ ]:
fit_004 = global_fit.recordings[0]
fractions = fit_004.component_fractions
time_1s = np.arange(fractions.shape[0]) + 0.5

fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(
    time_1s,
    fractions[:, 0],
    label=f"Short component ({global_fit.lifetimes[0]:.3f} ns)",
)

ax.plot(
    time_1s,
    fractions[:, 1],
    label=f"Long component ({global_fit.lifetimes[1]:.3f} ns)",
)

ax.set_xlabel("Recording time (s)")
ax.set_ylabel("Component photon fraction")
ax.set_ylim(0, 1)
ax.set_title(recording.path.name)
ax.legend()

plt.show()

In [ ]:
residual_background_counts = fit_004.coefficients[:, -1]

plt.figure(figsize=(13, 3))
plt.plot(time_1s, residual_background_counts)
plt.xlabel("Recording time (s)")
plt.ylabel("Fitted residual-background counts")
plt.title("Local residual background")
plt.show()

In [ ]:
print("Index:", recording_index)
print("File:", recording.path.name)

In [ ]:
fit_result = global_fit.recordings[0]

short_fraction = fit_result.component_fractions[:, 0]
long_fraction = fit_result.component_fractions[:, 1]

component_counts = fit_result.component_counts.sum(axis=1)
background_counts = fit_result.coefficients[:, -1]

background_fraction = (
    background_counts
    / (background_counts + component_counts)
)

raw_intensity_1s = bin_curves(
    recording.data[:, :, 0].sum(axis=0)[None, :],
    group_size=10,
    reducer="sum",
)[0]

print(
    "Correlation: long fraction vs background counts:",
    np.corrcoef(long_fraction, background_counts)[0, 1],
)

print(
    "Correlation: long fraction vs background fraction:",
    np.corrcoef(long_fraction, background_fraction)[0, 1],
)

print(
    "Correlation: background vs raw intensity:",
    np.corrcoef(background_counts, raw_intensity_1s)[0, 1],
)

In [ ]:
def zscore(values):
    values = np.asarray(values)
    return (values - values.mean()) / values.std()

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    time_1s,
    zscore(long_fraction),
    label="Long-lifetime fraction",
)

ax.plot(
    time_1s,
    zscore(background_fraction),
    label="Residual-background fraction",
    alpha=0.8,
)

ax.plot(
    time_1s,
    zscore(raw_intensity_1s),
    label="Raw intensity",
    alpha=0.7,
)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Recording time (s)")
ax.set_ylabel("Standardized value")
ax.legend()

plt.show()

In [ ]:
from scipy.optimize import nnls
from iflip3.models import design_matrix

group_size = 10

# double_fit_bg.background is the background per lifetime bin in the
# aggregate of all 3,000 samples. Scale it to one 10-sample temporal bin.
fixed_background_per_bin = (
    double_fit_bg.background
    * group_size
    / recording.n_samples
)

print(
    "Fixed residual background per lifetime bin:",
    fixed_background_per_bin,
)

In [ ]:
target_curves = (
    binned_decay_for_fit
    - fixed_background_per_bin
)

In [ ]:
basis = design_matrix(
    fit_time,
    lifetimes=double_fit_bg.lifetimes,
    t0=double_fit_bg.t0,
    irf_sigma=double_fit_bg.irf_sigma,
    pulse_interval=recording.header.pulse_interval_ns,
    normalize="area",
    include_background=False,
)

In [ ]:
component_coefficients = np.vstack([
    nnls(basis, target_curves[:, column])[0]
    for column in range(target_curves.shape[1])
])

component_fractions = (
    component_coefficients
    / component_coefficients.sum(axis=1, keepdims=True)
)

short_fraction_fixed = component_fractions[:, 0]
long_fraction_fixed = component_fractions[:, 1]

fitted_curves = basis @ component_coefficients.T
target_residuals = target_curves - fitted_curves

# Mean residual across lifetime bins—not included as a fitted parameter.
residual_offset = target_residuals.mean(axis=0)

In [ ]:
time_1s = np.arange(long_fraction_fixed.size) + 0.5

fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(
    time_1s,
    short_fraction_fixed,
    label=f"Short component ({double_fit_bg.lifetimes[0]:.3f} ns)",
)

ax.plot(
    time_1s,
    long_fraction_fixed,
    label=f"Long component ({double_fit_bg.lifetimes[1]:.3f} ns)",
)

ax.set_xlabel("Recording time (s)")
ax.set_ylabel("Component photon fraction")
ax.set_ylim(0, 1)
ax.set_title(recording.path.name)
ax.legend()

plt.show()

In [ ]:
print(
    "Long fraction vs residual offset:",
    np.corrcoef(long_fraction_fixed, residual_offset)[0, 1],
)

print(
    "Long fraction vs raw intensity:",
    np.corrcoef(long_fraction_fixed, raw_intensity_1s)[0, 1],
)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(
    time_1s,
    zscore(long_fraction_fixed),
    label="Long-lifetime fraction",
)

ax.plot(
    time_1s,
    zscore(residual_offset),
    label="Unfitted residual offset",
)

ax.plot(
    time_1s,
    zscore(raw_intensity_1s),
    label="Raw intensity",
    alpha=0.7,
)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Recording time (s)")
ax.set_ylabel("Standardized value")
ax.legend()

plt.show()

In [ ]:
short_counts = component_coefficients[:, 0]
long_counts = component_coefficients[:, 1]

fitted_sensor_intensity = short_counts + long_counts
long_photon_fraction = long_counts / fitted_sensor_intensity

In [ ]:
baseline = time_1s < 50

intensity_baseline = np.median(
    fitted_sensor_intensity[baseline]
)

fraction_baseline = np.median(
    long_photon_fraction[baseline]
)

delta_f_over_f = (
    fitted_sensor_intensity / intensity_baseline - 1
)

delta_long_fraction = (
    long_photon_fraction - fraction_baseline
)

fig, axes = plt.subplots(
    2,
    1,
    figsize=(13, 7),
    sharex=True,
    constrained_layout=True,
)

axes[0].plot(
    time_1s,
    100 * delta_f_over_f,
    color="tab:green",
)

axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("Fitted intensity ΔF/F (%)")

axes[1].plot(
    time_1s,
    delta_long_fraction,
    color="tab:orange",
)

axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Δ long photon fraction")
axes[1].set_xlabel("Recording time (s)")

plt.show()

In [ ]:
plt.figure(figsize=(6, 5))

points = plt.scatter(
    delta_long_fraction,
    100 * delta_f_over_f,
    c=time_1s,
    s=18,
    cmap="viridis",
)

plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Δ long photon fraction")
plt.ylabel("Fitted intensity ΔF/F (%)")
plt.colorbar(points, label="Recording time (s)")
plt.show()

In [ ]:
from scipy.stats import spearmanr

pearson = np.corrcoef(
    delta_long_fraction,
    delta_f_over_f,
)[0, 1]

spearman = spearmanr(
    delta_long_fraction,
    delta_f_over_f,
).statistic

print("Pearson correlation:", pearson)
print("Spearman correlation:", spearman)

In [ ]:
marker_values, marker_counts = np.unique(
    recording.marks,
    return_counts=True,
)

list(zip(marker_values, marker_counts))

In [ ]:
fig, axes = plt.subplots(
    3,
    1,
    figsize=(13, 8),
    sharex=True,
    constrained_layout=True,
)

axes[0].plot(
    time_1s,
    100 * delta_f_over_f,
    color="tab:green",
)

axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("Intensity ΔF/F (%)")

axes[1].plot(
    time_1s,
    delta_long_fraction,
    color="tab:orange",
)

axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Δ long fraction")

axes[2].step(
    recording.sample_time,
    recording.marks,
    where="mid",
    color="tab:purple",
)

axes[2].set_ylabel("Marker")
axes[2].set_xlabel("Recording time (s)")

plt.show()

In [ ]:
mpet_1s = bin_curves(
    mpet[:, 0][None, :],
    group_size=10,
    reducer="mean",
)[0]

baseline = time_1s < 50

delta_mpet = (
    mpet_1s
    - np.median(mpet_1s[baseline])
)

delta_long_fraction = (
    long_fraction_fixed
    - np.median(long_fraction_fixed[baseline])
)

intensity_dff = (
    fitted_sensor_intensity
    / np.median(fitted_sensor_intensity[baseline])
    - 1
)

In [ ]:
print(
    "Long fraction vs MPET:",
    np.corrcoef(delta_long_fraction, delta_mpet)[0, 1],
)

print(
    "Long fraction vs intensity:",
    np.corrcoef(delta_long_fraction, intensity_dff)[0, 1],
)

print(
    "MPET vs intensity:",
    np.corrcoef(delta_mpet, intensity_dff)[0, 1],
)

In [ ]:
fig, axes = plt.subplots(
    3,
    1,
    figsize=(13, 9),
    sharex=True,
    constrained_layout=True,
)

axes[0].plot(
    time_1s,
    delta_long_fraction,
    color="tab:orange",
)

axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_ylabel("Δ long fraction")
axes[0].set_title("Primary lifetime readout")

axes[1].plot(
    time_1s,
    delta_mpet,
    color="tab:blue",
)

axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_ylabel("Δ MPET (ns)")
axes[1].set_title("Model-independent lifetime readout")

axes[2].plot(
    time_1s,
    100 * intensity_dff,
    color="0.4",
)

axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_ylabel("Intensity ΔF/F (%)")
axes[2].set_xlabel("Recording time (s)")
axes[2].set_title("Intensity QC channel")

plt.show()

In [ ]:
print(
    "Long fraction vs MPET:",
    np.corrcoef(delta_long_fraction, delta_mpet)[0, 1],
)

print(
    "Long fraction vs intensity:",
    np.corrcoef(delta_long_fraction, intensity_dff)[0, 1],
)

print(
    "MPET vs intensity:",
    np.corrcoef(delta_mpet, intensity_dff)[0, 1],
)

In [ ]:
def robust_noise(values):
    values = np.asarray(values)
    center = np.median(values)
    return 1.4826 * np.median(np.abs(values - center))

baseline = time_1s < 50

fraction_noise = robust_noise(
    delta_long_fraction[baseline]
)

mpet_noise = robust_noise(
    delta_mpet[baseline]
)

intensity_noise = robust_noise(
    intensity_dff[baseline]
)

print("Long-fraction baseline noise:", fraction_noise)
print("MPET baseline noise:", mpet_noise, "ns")
print("Intensity baseline noise:", 100 * intensity_noise, "%")

print(
    "Long-fraction peak / baseline noise:",
    np.max(delta_long_fraction) / fraction_noise,
)

print(
    "MPET peak / baseline noise:",
    np.max(delta_mpet) / mpet_noise,
)